# colab_02_supervised — learned P(fail|x) on FREE T4 (RQ1)

Gate: asserts DECISION_01 calibration hash. Compares heuristic d(x) vs LogReg vs XGBoost vs DistilBERT-frozen.

Output: `router_supervised.pt`, `isotonic.json`, `DECISION_02.json` (freezes best encoder).

In [ ]:
# 0. Setup — self-recovering: finds repo, unzips upload, or clones (public or private via token)
import os, sys, json, hashlib, subprocess, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE = Path('/content/drive/MyDrive/Research/colab_ckpt')
    DRIVE.mkdir(parents=True, exist_ok=True)
    print('Drive:', DRIVE)
except Exception as e:
    print('No Colab Drive (local run?):', e)
    DRIVE = Path('./colab_ckpt')
    DRIVE.mkdir(parents=True, exist_ok=True)

def _find_backend():
    """Return BACKEND dir (containing src/gateway/service.py) or None."""
    cands = [Path('/content/code-mixed-gateway/backend'), Path('/content/Research/backend'),
             Path('/content/backend'), Path.cwd()/'backend', Path.cwd()/'code-mixed-gateway'/'backend']
    try:
        for p in Path('/content').iterdir():
            if p.is_dir():
                cands += [p, p/'backend']
    except Exception: pass
    for c in cands:
        if (c/'src'/'gateway'/'service.py').exists(): return c
    # deep search one level for upload_tmp style dirs
    try:
        for p in Path('/content').glob('*/**/service.py'):
            if p.parent.parent.name == 'gateway' and p.parent.name == 'gateway':
                be = p.parent.parent.parent  # .../src/gateway/service.py -> backend?
                # walk up to dir containing 'src'
                q = p.parent
                while q != Path('/content') and q.name != 'src': q = q.parent
                be = q.parent
                if (be/'src'/'gateway'/'service.py').exists(): return be
    except Exception: pass
    return None

def _try_unzip_uploads():
    roots = [Path('/content')]
    try: roots.append(DRIVE)
    except Exception: pass
    for root in roots:
        try: zips = list(root.glob('backend*.zip'))
        except Exception: continue
        for z in zips:
            dest = Path('/content/repo_upload')
            if (dest/'backend'/'src'/'gateway'/'service.py').exists(): return True
            print(f'unzipping {z} -> {dest} ...')
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z) as zf: zf.extractall(dest)
            # handle zip containing backend/ at top or nested one level
            if not (dest/'backend'/'src'/'gateway'/'service.py').exists():
                for sub in dest.iterdir():
                    if (sub/'backend'/'src'/'gateway'/'service.py').exists():
                        import shutil
                        shutil.move(str(sub/'backend'), str(dest/'backend_tmp'))
                        shutil.move(str(dest/'backend_tmp'), str(dest/'backend'))
                        break
            print('unzip done')
            return True
    return False

def _try_clone():
    dest = Path('/content/code-mixed-gateway')
    if dest.exists(): return dest.exists()
    token = ''
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN') or ''
    except Exception: token = os.environ.get('GITHUB_TOKEN','') or ''
    url = ('https://'+token+'@github.com/tusharsaharan/code-mixed-gateway.git') if token else 'https://github.com/tusharsaharan/code-mixed-gateway.git'
    print('cloning (token=%s) ...' % ('yes' if token else 'no'))
    r = subprocess.run(['git','clone',url,str(dest)], capture_output=True, text=True)
    print('clone rc=', r.returncode)
    if r.returncode != 0: print((r.stderr or '')[-500:])
    return dest.exists()

BACKEND = _find_backend()
if BACKEND is None and _try_unzip_uploads(): BACKEND = _find_backend()
if BACKEND is None and _try_clone(): BACKEND = _find_backend()
print('BACKEND =', BACKEND)
assert BACKEND is not None, ('REPO NOT FOUND. Fastest fix (2 min): on your PC zip the backend folder '
    '(Research/backend -> backend.zip), drag backend.zip into Colab Files panel (folder icon, left), then '
    'Cell > Run this cell again. Alternative: make the GitHub repo public, or add a GITHUB_TOKEN secret.')
REPO = BACKEND.parent
DATA = BACKEND/'data'
print('REPO =', REPO, '| DATA exists:', DATA.exists())

%pip install -q pydantic pydantic-settings numpy httpx 2>&1 | tail -1
src = str(BACKEND/'src')
if src not in sys.path: sys.path.insert(0, src)
import gateway
print('import gateway OK from', gateway.__file__)


In [ ]:
# 1. Features (all local, free): heuristic 5 + MiniLM-384 (frozen)
import json, numpy as np
from gateway.modules.m4_router.difficulty import DifficultyScorer
from gateway.modules.m1_pipeline.hinglish import code_mix_ratio
from gateway.tokenizer import TokenCounter
scorer = DifficultyScorer(); counter = TokenCounter()
rows = [json.loads(l) for l in open(DRIVE/'calibration_real.jsonl', encoding='utf-8')]
lab = {json.loads(l)['id']: json.loads(l) for l in open(DRIVE/'groq_labels.jsonl', encoding='utf-8')}
splits = json.loads((DRIVE/'splits.json').read_text())
by_id = {r['id']: r for r in rows}

def feats(rid):
    t = lab[rid]['text']; f = scorer.features(t)
    return [f.code_mix_ratio, f.entity_density, min(1.0, f.math_marker_count/3.0), min(1.0, f.char_count/400.0), len(t.split())/50.0]

from sentence_transformers import SentenceTransformer
enc = SentenceTransformer('all-MiniLM-L6-v2')  # 22M, T4-safe, free
ids_all = [r['id'] for r in rows]
emb = enc.encode([lab[i]['text'] for i in ids_all], batch_size=64, show_progress_bar=True, normalize_embeddings=True)
import numpy as np
FEAT = {rid: np.concatenate([np.array(feats(rid), dtype=np.float32), emb[k]]) for k, rid in enumerate(ids_all)}
Y = {r['id']: (0 if r['cheap_success'] else 1) for r in rows}  # y=1 means cheap FAILS
print('feat dim:', len(next(iter(FEAT.values()))))

In [ ]:
# 2. Train LogReg / XGB on train, validate on calib. DistilBERT-head fine-tune (1-3 epochs, fp16) optional if time.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from gateway.modules.m11_calibration.ece import expected_calibration_error
import numpy as np
tr, ca = splits['train'], splits['calib']
Xtr = np.stack([FEAT[i] for i in tr]); ytr = np.array([Y[i] for i in tr])
Xca = np.stack([FEAT[i] for i in ca]); yca = np.array([Y[i] for i in ca])
# heuristic baseline: nonconformity score as P(fail)
h_tr = np.array([by_id[i]['nonconformity'] for i in tr]); h_ca = np.array([by_id[i]['nonconformity'] for i in ca])
print('heuristic AUROC train/calib:', round(roc_auc_score(ytr, h_tr),4), round(roc_auc_score(yca, h_ca),4))

clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
p_ca = clf.predict_proba(Xca)[:,1]
print('logreg AUROC calib:', round(roc_auc_score(yca, p_ca),4), 'AUPRC:', round(average_precision_score(yca, p_ca),4))
print('logreg ECE calib:', round(expected_calibration_error(1-p_ca, [bool(1-y) for y in yca]),4))

import xgboost as xgb
gx = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, n_jobs=4, eval_metric='logloss')
gx.fit(Xtr, ytr)
g_ca = gx.predict_proba(Xca)[:,1]
print('xgb AUROC calib:', round(roc_auc_score(yca, g_ca),4), 'ECE:', round(expected_calibration_error(1-g_ca, [bool(1-y) for y in yca]),4))

# Save both; NB03/NB04 picks winner by AUROC on calib (no test touch!)
import pickle
winner = 'xgb' if roc_auc_score(yca, g_ca) >= roc_auc_score(yca, p_ca) else 'logreg'
pickle.dump(clf, open(DRIVE/'logreg.pkl','wb')); gx.save_model(str(DRIVE/'xgb.json'))
(DRIVE/'DECISION_02.json').write_text(json.dumps({'stage':'02_supervised','winner':winner,'auroc_heuristic':round(float(roc_auc_score(yca,h_ca)),4),'auroc_logreg':round(float(roc_auc_score(yca,p_ca)),4),'auroc_xgb':round(float(roc_auc_score(yca,g_ca)),4),'exit_pass': bool(max(roc_auc_score(yca,p_ca),roc_auc_score(yca,g_ca)) > roc_auc_score(yca,h_ca)+0.03)}, indent=2))
print('winner:', winner)